In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open('names.txt', 'r').read().splitlines()

In [4]:
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [5]:
# buld the vocabulary of characters and mappings to/fro integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}


In [14]:
# build the training dataset
block_size = 3 # the context length: how many characters do we take to predict the next one?
X, Y = [], [] # the input and the labels array, it's the actual dataset we're building
for w in words[:5]:
    print(w)
    context = [0] * block_size # Show that we're at the beginning of a word [0, 0, 0]
    for ch in w + '.': # loop through the word and add a dot at the end of the word
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '---->', itos[ix])
        context = context[1:] + [ix] # crop the first character then append ix
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [15]:
X.shape

torch.Size([32, 3])

In [16]:
Y.shape

torch.Size([32])

In [17]:
# Create a lookup table or embedding layer
C = torch.randn((27,2))
emb = C[X] # mapping the inputs to the lookup table
emb.shape

torch.Size([32, 3, 2])

In [34]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [35]:
# torch.cat(torch.unbind(emb, 1), 1).shape # collapse of the dimernsions is inefficient since it creates new memory
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # we use -1 or emb.shape[0] to infer what the value should be

In [36]:
h

tensor([[-0.9904,  0.7983, -0.5094,  ..., -0.9990, -0.9999,  0.9248],
        [ 0.9998, -0.9867,  0.8476,  ..., -0.9963, -0.9963,  0.9672],
        [ 1.0000,  0.5408, -0.9917,  ..., -0.1884, -1.0000, -0.9566],
        ...,
        [ 0.9896, -0.9977,  0.8537,  ...,  0.9935,  0.6935,  0.9950],
        [ 0.5128,  0.9964, -0.9653,  ...,  0.9983, -0.9999, -0.4946],
        [-0.9899, -0.9999,  0.9974,  ...,  0.1811,  0.9996,  0.9970]])

In [37]:
h.shape

torch.Size([32, 100])

In [54]:
w2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [55]:
logits = h @ w2 + b2

In [56]:
logits.shape

torch.Size([32, 27])

In [57]:
counts = logits.exp()

In [58]:
prob = counts/ counts.sum(1, keepdim=True)

In [59]:
prob.shape

torch.Size([32, 27])

In [63]:
loss = -prob[torch.arange(32), Y].log().mean() # the negative log likelihood loss
loss

tensor(17.2359)

In [ ]:
# ----- CLEAN UP OF THE CODE

In [71]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [72]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
w1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
w2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, w1, b1, w2, b2]

In [73]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [74]:
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h @ w2 + b2 # (32, 27)
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(20.9608)